# 第 21 章 · online softmax 与算子融合：少读一遍显存就是快

本章回答：softmax 为什么要先求最大值？标准写法要把数据读几遍？online softmax 怎么边读边更新最大值和分母、只读一遍？什么是算子融合，为什么"两个 memory-bound 的 kernel 合成一个"几乎能快一倍？

配套讲义：仓库里 `cuda-programming/notes/lessons/21-online-softmax-fusion.md`（讲解、动画、练习都在讲义里）。对应原文：FlashAttention-2 论文 §2.3.1 online softmax（`materials/text/flashattention-2.txt` 第 138–200 行）；Milakov & Gimelshein「Online normalizer calculation for softmax」（2018，〔旧知〕）；Quack《Getting Memory-bound Kernels to Speed-of-Light》（在线）；Liger Kernel（[04 调研](../04-resources-and-optimization-survey.md) §3.3）；cutile-rs tutorial 05 softmax（`code/nvidia-cuda-rust/repo/`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import math

import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 溢出 | $e^{100}$ | `np.exp(np.float16(100))` | inf；减最大值后正常 |
| 第三步 online | $\ell \leftarrow \ell e^{m - m'} + e^{x - m'}$ | `online_ml([1, 3, 2])` | (1, 1) → (3, 1.1353) → (3, 1.5032) |
| 第三步 直接算 | $\sum e^{x_j - 3}$ | `sum(math.exp(v - 3) for v in [1, 3, 2])` | 1.5032 |
| 第三步 分块合并 | $\ell = \ell_1 e^{m_1 - m} + \ell_2 e^{m_2 - m}$ | `merge(online_ml([1, 3])[-1], online_ml([2])[-1])` | = 一次读完的 (3, 1.5032) |
| 第四步 融合 | 4 次 vs 2 次读写 | `fusion_bytes(100_000_000)` | 800 MB vs 400 MB |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `fusion_bytes`**（来自 `ch21_online_softmax.py`）：y = gelu(x + bias)：两个 kernel 读 x、写 t、读 t、写 y；融合后只读 x、写 y。

In [ ]:
def fusion_bytes(n: int, bytes_per: int = 2) -> tuple[int, int]:
    """y = gelu(x + bias)：两个 kernel 读 x、写 t、读 t、写 y；融合后只读 x、写 y。"""
    return 4 * n * bytes_per, 2 * n * bytes_per

**函数 `merge`**（来自 `ch21_online_softmax.py`）：两块的 (m, ℓ) 合并：m = max(m1, m2)，ℓ = ℓ1 e^(m1 − m) + ℓ2 e^(m2 − m)。

In [ ]:
def merge(a: tuple[float, float], b: tuple[float, float]) -> tuple[float, float]:
    """两块的 (m, ℓ) 合并：m = max(m1, m2)，ℓ = ℓ1 e^(m1 − m) + ℓ2 e^(m2 − m)。"""
    m = max(a[0], b[0])
    return m, a[1] * math.exp(a[0] - m) + b[1] * math.exp(b[0] - m)

**函数 `online_ml`**（来自 `ch21_online_softmax.py`）：一遍扫过去维护 (m, ℓ)：遇到更大的数，旧账乘 e^(m_old − m_new)。返回每读一个数之后的 (m, ℓ)。

In [ ]:
def online_ml(xs) -> list[tuple[float, float]]:
    """一遍扫过去维护 (m, ℓ)：遇到更大的数，旧账乘 e^(m_old − m_new)。返回每读一个数之后的 (m, ℓ)。"""
    m, l = -math.inf, 0.0
    trace = []
    for x in xs:
        m_new = max(m, x)
        l = l * math.exp(m - m_new) + math.exp(x - m_new)   # m = −∞ 时 exp(−∞) = 0
        m = m_new
        trace.append((m, l))
    return trace

**函数 `softmax_naive`**（来自 `ch21_online_softmax.py`）：直接 e^x / Σe^x：数大了会溢出。

In [ ]:
def softmax_naive(x: np.ndarray) -> np.ndarray:
    """直接 e^x / Σe^x：数大了会溢出。"""
    e = np.exp(x)
    return e / e.sum()

**函数 `softmax_safe`**（来自 `ch21_online_softmax.py`）：先减最大值：所有指数 ≤ 0，不会溢出；结果不变。

In [ ]:
def softmax_safe(x: np.ndarray) -> np.ndarray:
    """先减最大值：所有指数 ≤ 0，不会溢出；结果不变。"""
    e = np.exp(x - x.max())
    return e / e.sum()

**函数 `softmax_two_pass`**（来自 `ch21_online_softmax.py`）：online 版：第一遍得到 (m, ℓ)，第二遍输出 e^(x − m) / ℓ。

In [ ]:
def softmax_two_pass(x: np.ndarray) -> np.ndarray:
    """online 版：第一遍得到 (m, ℓ)，第二遍输出 e^(x − m) / ℓ。"""
    m, l = online_ml(x)[-1]
    return np.exp(x - m) / l

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 e^100 放不进 FP16（最大约 65504）")
with np.errstate(over="ignore", invalid="ignore"):
    print("  np.exp(float16(100)) =", np.exp(np.float16(100)), "；float32:", np.exp(np.float32(100)))
    x = np.array([100, 101, 102], dtype=np.float16)
    print("  直接算 softmax：", softmax_naive(x), "；先减最大值：", softmax_safe(x))

In [ ]:
print("\n第三步 online softmax，x = [1, 3, 2]")
for v, (m, l) in zip([1, 3, 2], online_ml([1, 3, 2])):
    print(f"  读到 {v}: m = {m}, ℓ = {l:.4f}")
direct = sum(math.exp(v - 3) for v in [1, 3, 2])
print(f"  直接算 Σ e^(x − 3) = {direct:.4f}")
xs = np.array([1.0, 3.0, 2.0])
print("  两遍版 softmax 和三遍版一样：", np.allclose(softmax_two_pass(xs), softmax_safe(xs)))

In [ ]:
print("\n第三步 分块合并：[1, 3] 和 [2] 各算各的，再合并")
print("  ", merge(online_ml([1, 3])[-1], online_ml([2])[-1]), "= 一次读完的", online_ml([1, 3, 2])[-1])

In [ ]:
print("\n第二、三步 访存次数：三遍读 + 一遍写 = 4 次；online 两遍读 + 一遍写 = 3 次")

In [ ]:
print("\n第四步 1 亿个 BF16 做 y = gelu(x + bias)")
two, fused = fusion_bytes(100_000_000)
print(f"  两个 kernel：{two / 1e6:.0f} MB；融合：{fused / 1e6:.0f} MB；耗时约为 {fused / two:.0%}")

## 3. 代码里才看得到的

1. **`math.exp(-math.inf)` 等于 0**：所以第一次循环时 `l * exp(m - m_new)` 自动是 0，不用单独写"第一次"的分支。
2. **溢出时 NumPy 给的是 inf 和 nan，不报错**：`softmax_naive` 在 FP16 下得到 `[nan nan nan]`（inf ÷ inf）；代码用 `np.errstate` 把警告关掉了，GPU 上同样是悄悄变成 nan。
3. **`merge` 满足结合律**：`merge(merge(a, b), c) == merge(a, merge(b, c))`，所以能按任意分组并行归约（第 10–11 章）。

## 4. 自己改一改

- 把 `online_ml` 的输入换成一长串随机数，和 `np.log(np.exp(x).sum())` 对比 `m + log ℓ`。
- 用 `merge` 把一串数分成 3 块、4 块分别算再合并，确认结果不变。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> softmax 先减最大值，指数不会爆；
> 三遍变两遍：边读边改账，遇到更大的，旧账乘 e 的差；
> (m, ℓ) 能合并，就能分块、能并行；
> 融合：中间结果别落地，两个 memory-bound 合一个，时间砍一半；
> 光速两件事：合并读写，按行长选归约层级。